# T3P Packet Inspector

In [12]:
import struct
import re
import os
import mmap
import numpy as np


def inspect_and_parse_t3p(filepath, start_packet, end_packet):
    print(f"--- T3P EXACT PACKET INSPECTOR & PARSER ---")
    print(f"File: {os.path.basename(filepath)}")
    print(f"Printing records #{start_packet} through #{end_packet}...")
    print(f"Scanning the entire file to build NumPy arrays...\n")

    # This strict regex guarantees exactly 6 numbers separated by 5 tabs, ending in 10
    hw_trigger_pattern = re.compile(rb'\d+\t\d+\t\d+\t\d+\t\d+\t10\r?\n')

    # Lists to accumulate data before NumPy conversion
    photon_list = []
    trigger_list = []

    with open(filepath, 'rb') as f:
        # Use memory-mapping to scan massive files instantly
        mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)

        # Pre-scan for all ASCII HW Trigger injections
        hw_trigger_traps = list(hw_trigger_pattern.finditer(mm))
    
        offset = 0
        record_count = 0
        header_printed = False


        # Global Counters for the entire file
        chip_0_hits = 0  
        chip_1_hits = 0  

        # Loop over the ENTIRE file
        while offset < len(mm):

            # 1. CHECK FOR ASCII HW TRIGGER INJECTIONS
            if hw_trigger_traps and offset <= hw_trigger_traps[0].start() < offset + 16:
                trap = hw_trigger_traps.pop(0)

                # Jump over any corrupted/cut-off bytes directly to the text
                offset = trap.start()

                # The text string itself counts as one record in the sequence
                record_count += 1
                text_bytes = mm[trap.start() : trap.end()]
                text_str = text_bytes.decode('ascii', errors='ignore').strip()

                # Split the string by tabs, convert to integers, and save as a tuple!
                parts = text_str.split('\t')
                if len(parts) == 6:
                    trigger_list.append(tuple(map(int, parts)))

                # Handle the printing if it falls in the user's requested range
                if start_packet <= record_count <= end_packet:
                    display_str = text_str.replace('\t', ' \\t ')
                    print(f"\n[{record_count:08d}] [OFFSET: {trap.start():08X}] ⚡ HW TRIGGER (TEXT) | Length: {len(text_bytes):02d} bytes | Text: '{display_str}'\n")
                    header_printed = False # Reset header so it reprints after the text interruption

                # Resume standard 16-byte reading immediately after the text ends
                offset = trap.end()
                continue

            # 2. READ THE STANDARD 16-BYTE BINARY PACKET (PHOTON HITS)
            if offset + 16 > len(mm):
                break

            packet = mm[offset : offset + 16]

            # Unpack all 5 variables from the C-Struct
            matrixIdx, toa, overflow, ftoa, tot = struct.unpack('<IQBBH', packet)
            record_count += 1

            # Store the photon hit as a tuple
            photon_list.append((matrixIdx, toa, overflow, ftoa, tot))
            # Global Tally logic
            if overflow == 0:
                chip_0_hits += 1
            else:
                chip_1_hits += 1

            # 3. PRINT THE PACKET ONLY IF IT FALLS IN THE RANGE
            if start_packet <= record_count <= end_packet:

                if not header_printed:
                    print(f"{'RECORD #':<10} | {'BYTE OFFSET':<11} | {'PACKET TYPE':<14} | {'matrixIdx':<10} | {'ToA':<12} | {'ToT':<5} | {'fToA':<4} | {'Overflow (Chip)'}")
                    print("-" * 95)
                    header_printed = True

                print(f"[{record_count:08d}] | {offset:08X}    | 🔵 PHOTON HIT   | {matrixIdx:<10} | {toa:<12} | {tot:<5} | {ftoa:<4} | {overflow}")

            offset += 16

        mm.close()

    # ==========================================
    # NUMPY ARRAY CONVERSION
    # ==========================================
    photon_dtype = np.dtype([
        ('matrixIdx', np.uint32),
        ('toa', np.uint64),
        ('overflow', np.uint8),
        ('ftoa', np.uint8),
        ('tot', np.uint16)

    ])

    trigger_dtype = np.dtype([
        ('record_idx', np.uint32),
        ('matrixIdx',  np.uint32),
        ('toa',        np.uint64),
        ('tot',        np.uint32),
        ('trigger_id', np.uint32),
        ('overflow',   np.uint32)
    ])

    photon_array = np.array(photon_list, dtype=photon_dtype)
    trigger_array = np.array(trigger_list, dtype=trigger_dtype)

    print("\n--- INSPECTION COMPLETE ---")
    print(f"► Total HW triggers (Text):           {len(trigger_array)}")
    print(f"► Total photon hits (Binary):         {len(photon_array)}")
    print(f"  ├─ Hits on Chip 0 (overflow == 0):  {chip_0_hits}")
    print(f"  └─ Hits on Chip 1+ (overflow != 0): {chip_1_hits}")
    print("\n--- MATRIX INDEX ANALYSIS ---")

    if len(photon_array) > 0:
        max_idx = np.max(photon_array['matrixIdx'])
        print(f"► Maximum matrixIdx found: {max_idx}")

        # 256 * 256 = 65,536 pixels per chip. Indices run 0 to 65,535.
        if max_idx > 65535:
            print("  └─ Conclusion: Index exceeds 65,535. The camera uses a GLOBAL indexing scheme across both chips (e.g., 256x512).")
        else:
            print("  └─ Conclusion: Index is <= 65,535. The camera uses SEPARATE indexing per chip (resets to 0 for the second chip).")
    else:
        print("► No photon hits found to analyze.")

    print("\n--- OVERFLOW ANALYSIS ---")  
    if len(photon_array) > 0:
        max_overflow = np.max(photon_array['overflow'])
        print(f"► Maximum overflow found: {max_overflow}")

    if len(trigger_array) > 0:
        max_idx = np.max(trigger_array['record_idx'])
        print(f"\n► Triggers Array Length: {len(trigger_array)}")
        if len(trigger_array) == max_idx + 1:
            print("  └─ Status: PERFECT MATCH! No fake triggers were captured.")
        else:
            print("  └─ Status: WARNING! The array length doesn't match the final index. Some triggers are missing or skipped.")    

    return photon_array, trigger_array


# ==========================================
# EXECUTION
# ==========================================

t3p_file = r"G:\האחסון שלי\X-Ray-IFM\Test Files\Sync_test\25.6\both_detectors_25.6_r0.t3p"

start = 0
end = 5

# Capture the returned arrays

photons, triggers = inspect_and_parse_t3p(t3p_file, start, end) 

--- T3P EXACT PACKET INSPECTOR & PARSER ---
File: both_detectors_25.6_r0.t3p
Printing records #0 through #5...
Scanning the entire file to build NumPy arrays...


[00000001] [OFFSET: 00000000] ⚡ HW TRIGGER (TEXT) | Length: 18 bytes | Text: '0 \t 0 \t 62 \t 0 \t 49152 \t 10'


[00000002] [OFFSET: 00000012] ⚡ HW TRIGGER (TEXT) | Length: 14 bytes | Text: '1 \t 0 \t 62 \t 0 \t 0 \t 10'

RECORD #   | BYTE OFFSET | PACKET TYPE    | matrixIdx  | ToA          | ToT   | fToA | Overflow (Chip)
-----------------------------------------------------------------------------------------------
[00000003] | 00000020    | 🔵 PHOTON HIT   | 78124      | 426971       | 9     | 21   | 1
[00000004] | 00000030    | 🔵 PHOTON HIT   | 77611      | 426971       | 15    | 19   | 1
[00000005] | 00000040    | 🔵 PHOTON HIT   | 77612      | 426971       | 15    | 18   | 1

--- INSPECTION COMPLETE ---
► Total HW triggers (Text):           244
► Total photon hits (Binary):         109719
  ├─ Hits on Chip 0 (overflow ==

## T3P inspection with html table

In [13]:
import struct
import re
import os
import mmap
import numpy as np
import webbrowser

def inspect_and_parse_t3p(filepath, start_packet, end_packet):
    print(f"--- T3P EXACT PACKET INSPECTOR & PARSER ---")
    print(f"File: {os.path.basename(filepath)}")
    print(f"Printing records #{start_packet} through #{end_packet} to HTML...")
    print(f"Scanning the entire file to build NumPy arrays...\n")
    
    # This strict regex guarantees exactly 6 numbers separated by 5 tabs, ending in 10
    hw_trigger_pattern = re.compile(rb'\d+\t\d+\t\d+\t\d+\t\d+\t10\r?\n')
    
    # Lists to accumulate data before NumPy conversion
    photon_list = []
    trigger_list = []
    
    # --- HTML INITIALIZATION ---
    html_rows = []
    
    with open(filepath, 'rb') as f:
        # Use memory-mapping to scan massive files instantly
        mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        
        # Pre-scan for all ASCII HW Trigger injections
        hw_trigger_traps = list(hw_trigger_pattern.finditer(mm))
        
        offset = 0
        record_count = 0
        
        # Global Counters for the entire file
        chip_0_hits = 0  
        chip_1_hits = 0  
        
        # Loop over the ENTIRE file
        while offset < len(mm):
            
            # 1. CHECK FOR ASCII HW TRIGGER INJECTIONS
            if hw_trigger_traps and offset <= hw_trigger_traps[0].start() < offset + 16:
                trap = hw_trigger_traps.pop(0)
                
                # Jump over any corrupted/cut-off bytes directly to the text
                offset = trap.start()
                
                # The text string itself counts as one record in the sequence
                record_count += 1
                
                text_bytes = mm[trap.start() : trap.end()]
                text_str = text_bytes.decode('ascii', errors='ignore').strip()
                
                # Split the string by tabs, convert to integers
                parts = text_str.split('\t')
                if len(parts) == 6:
                    trigger_list.append(tuple(map(int, parts)))
                
                # Format for HTML output if in range
                if start_packet <= record_count <= end_packet:
                    hw_id, mat_idx, toa, tot, trig_id, overflow = parts if len(parts) == 6 else ("-", "-", "-", "-", "-", "-")
                    
                    html_rows.append(f"""
                    <tr class="hw-trigger">
                        <td>{record_count:08d}</td>
                        <td class="highlight-hw">{hw_id}</td>
                        <td>{trap.start():08X}</td>
                        <td>{len(text_bytes)}</td>
                        <td><b>⚡ HW TRIGGER</b></td>
                        <td>{mat_idx}</td>
                        <td>{toa}</td>
                        <td>{tot}</td>
                        <td>{trig_id}</td>
                        <td>{overflow}</td>
                    </tr>
                    """)
                
                # Resume standard 16-byte reading immediately after the text ends
                offset = trap.end()
                continue
                
            # 2. READ THE STANDARD 16-BYTE BINARY PACKET (PHOTON HITS)
            if offset + 16 > len(mm):
                break 
                
            packet = mm[offset : offset + 16]
            
            # Unpack all 5 variables from the C-Struct
            matrixIdx, toa, overflow, ftoa, tot = struct.unpack('<IQBBH', packet)
            record_count += 1
            
            # Store the photon hit as a tuple
            photon_list.append((matrixIdx, toa, overflow, ftoa, tot))
            
            # Global Tally logic
            if overflow == 0:
                chip_0_hits += 1
            else:
                chip_1_hits += 1
            
            # 3. FORMAT FOR HTML OUTPUT IF IN RANGE
            if start_packet <= record_count <= end_packet:
                html_rows.append(f"""
                <tr class="photon-hit">
                    <td>{record_count:08d}</td>
                    <td class="dim">-</td>
                    <td>{offset:08X}</td>
                    <td>16</td>
                    <td>🔵 PHOTON HIT</td>
                    <td>{matrixIdx}</td>
                    <td>{toa}</td>
                    <td>{tot}</td>
                    <td>{ftoa}</td>
                    <td>{overflow}</td>
                </tr>
                """)
                
            offset += 16
            
        mm.close()
    
    # ==========================================
    # HTML GENERATION
    # ==========================================
    html_content = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>T3P Packet Inspection</title>
        <style>
            body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f8f9fa; padding: 20px; color: #333; }}
            h2 {{ border-bottom: 2px solid #2c3e50; padding-bottom: 10px; color: #2c3e50; }}
            .summary {{ background: #fff; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin-bottom: 20px; }}
            table {{ border-collapse: collapse; width: 100%; background-color: #fff; box-shadow: 0 2px 8px rgba(0,0,0,0.1); font-family: monospace; font-size: 14px; }}
            th, td {{ border: 1px solid #e9ecef; padding: 12px 15px; text-align: center; }}
            th {{ background-color: #2c3e50; color: white; text-transform: uppercase; letter-spacing: 0.05em; font-size: 13px; }}
            tr:hover {{ background-color: #f1f3f5; }}
            .hw-trigger {{ background-color: #fff4e5; }} /* Light orange tint */
            .photon-hit {{ background-color: #ffffff; }} 
            .highlight-hw {{ font-weight: bold; color: #d35400; }}
            .dim {{ color: #adb5bd; }}
        </style>
    </head>
    <body>
        <h2>T3P Packet Inspection Report</h2>
        <div class="summary">
            <strong>File:</strong> {os.path.basename(filepath)}<br>
            <strong>Records Displayed:</strong> #{start_packet} to #{end_packet}
        </div>
        <table>
            <thead>
                <tr>
                    <th>RECORD #</th>
                    <th title="Only applies to HW Triggers">HW ID</th>
                    <th>BYTE OFFSET</th>
                    <th>LENGTH</th>
                    <th>PACKET TYPE</th>
                    <th>matrixIdx</th>
                    <th>ToA</th>
                    <th>ToT</th>
                    <th title="fToA for Photons / Trigger ID for HW Triggers">fToA / Trig ID</th>
                    <th>Overflow</th>
                </tr>
            </thead>
            <tbody>
                {"".join(html_rows)}
            </tbody>
        </table>
    </body>
    </html>
    """
    
    # Save the HTML to a file in the same directory as the .t3p file
    html_filepath = os.path.join(os.path.dirname(filepath), "inspection_report.html")
    with open(html_filepath, 'w', encoding='utf-8') as html_file:
        html_file.write(html_content)
        
    print(f"✅ HTML report generated successfully: {html_filepath}")
    webbrowser.open('file://' + os.path.realpath(html_filepath))

    # ==========================================
    # NUMPY ARRAY CONVERSION
    # ==========================================
    photon_dtype = np.dtype([
        ('matrixIdx', np.uint32),
        ('toa', np.uint64),
        ('overflow', np.uint8),
        ('ftoa', np.uint8),
        ('tot', np.uint16)
    ])
    
    trigger_dtype = np.dtype([
        ('record_idx', np.uint32),
        ('matrixIdx',  np.uint32),
        ('toa',        np.uint64),
        ('tot',        np.uint32),
        ('trigger_id', np.uint32), 
        ('overflow',   np.uint32)
    ])

    photon_array = np.array(photon_list, dtype=photon_dtype)
    trigger_array = np.array(trigger_list, dtype=trigger_dtype)
        
    print("\n--- INSPECTION COMPLETE ---")
    print(f"► Total HW triggers (Text):           {len(trigger_array)}")
    print(f"► Total photon hits (Binary):         {len(photon_array)}")
    print(f"  ├─ Hits on Chip 0 (overflow == 0):  {chip_0_hits}")
    print(f"  └─ Hits on Chip 1+ (overflow != 0): {chip_1_hits}")
    
    print("\n--- MATRIX INDEX ANALYSIS ---")
    if len(photon_array) > 0:
        max_idx = np.max(photon_array['matrixIdx'])
        print(f"► Maximum matrixIdx found: {max_idx}")
        
        # 256 * 256 = 65,536 pixels per chip. Indices run 0 to 65,535.
        if max_idx > 65535:
            print("  └─ Conclusion: Index exceeds 65,535. The camera uses a GLOBAL indexing scheme across both chips (e.g., 256x512).")
        else:
            print("  └─ Conclusion: Index is <= 65,535. The camera uses SEPARATE indexing per chip (resets to 0 for the second chip).")
    else:
        print("► No photon hits found to analyze.")

    print("\n--- OVERFLOW ANALYSIS ---")  
    if len(photon_array) > 0:
        max_overflow = np.max(photon_array['overflow'])
        print(f"► Maximum overflow found: {max_overflow}")

    if len(trigger_array) > 0:
        max_idx = np.max(trigger_array['record_idx'])
        print(f"\n► Triggers Array Length: {len(trigger_array)}")
        if len(trigger_array) == max_idx + 1:
            print("  └─ Status: PERFECT MATCH! No fake triggers were captured.")
        else:
            print("  └─ Status: WARNING! The array length doesn't match the final index. Some triggers are missing or skipped.")    

    return photon_array, trigger_array

# ==========================================
# EXECUTION
# ==========================================
t3p_file = r"G:\האחסון שלי\X-Ray-IFM\Test Files\Sync_test\25.6\both_detectors_25.6_r1.t3p"

start = 0
end = 50

# Capture the returned arrays
photons, triggers = inspect_and_parse_t3p(t3p_file, start, end)

--- T3P EXACT PACKET INSPECTOR & PARSER ---
File: both_detectors_25.6_r1.t3p
Printing records #0 through #50 to HTML...
Scanning the entire file to build NumPy arrays...

✅ HTML report generated successfully: G:\האחסון שלי\X-Ray-IFM\Test Files\Sync_test\25.6\inspection_report.html

--- INSPECTION COMPLETE ---
► Total HW triggers (Text):           244
► Total photon hits (Binary):         110264
  ├─ Hits on Chip 0 (overflow == 0):  55025
  └─ Hits on Chip 1+ (overflow != 0): 55239

--- MATRIX INDEX ANALYSIS ---
► Maximum matrixIdx found: 131070
  └─ Conclusion: Index exceeds 65,535. The camera uses a GLOBAL indexing scheme across both chips (e.g., 256x512).

--- OVERFLOW ANALYSIS ---
► Maximum overflow found: 1

► Triggers Array Length: 244
  └─ Status: PERFECT MATCH! No fake triggers were captured.


# .h5 Bins Inspector

In [ ]:
import h5py
import numpy as np
import os

def inspect_px5_h5(filepath, start_bin, end_bin):
    print(f"--- HDF5 PX5 DATA INSPECTOR ---")
    print(f"File: {os.path.basename(filepath)}")

    try:
        with h5py.File(filepath, 'r') as f:
            # 1. Read the Metadata attributes written by the DAQ script
            bin_s = f.attrs['bin_s']
            
            # 2. Access the main dataset
            if 'px5CountsPerBin' not in f:
                print("Error: Dataset 'px5CountsPerBin' not found in file.")
                return
                
            dset = f['px5CountsPerBin']
            total_bins = dset.shape[0]
            
            print(f"Total Bins   : {total_bins:,}")
            print(f"Bin Resolution: {bin_s * 1e6:.1f} µs ({bin_s} s)")
            print(f"File Duration: {total_bins * bin_s:.2f} seconds")
            
            # We can sum the entire array instantly to find the total hits
            print(f"Total Photons: {np.sum(dset):,}") 
            print("-" * 55)
            
            # 3. Validate user input range
            if start_bin < 0: start_bin = 0
            if end_bin >= total_bins: end_bin = total_bins - 1
            if start_bin > end_bin:
                print("Invalid range selected.")
                return
            
            print(f"Scanning bins #{start_bin} to #{end_bin}...\n")
            print(f"{'BIN INDEX':<12} | {'TIME (Seconds)':<15} | {'PHOTON COUNT'}")
            print("-" * 45)
            
            # 4. Extract only the specific slice of memory requested
            counts_in_range = dset[start_bin : end_bin + 1]
            
            for i, count in enumerate(counts_in_range):
                actual_bin = start_bin + i
                time_s = actual_bin * bin_s
                
                # Add a visual flag if a photon was actually detected in this bin
                if count > 0:
                    count_str = f"{count}  <-- 🟢 HIT"
                else:
                    count_str = str(count)
                    
                print(f"[{actual_bin:08d}]   | {time_s:<15.6f} | {count_str}")
                
    except Exception as e:
        print(f"Failed to read HDF5 file: {e}")

# ==========================================
# EXECUTION
# ==========================================
# Point this to your generated DAQ file
h5_file = r"C:\IFM\Sync_meas_23.6\sync_23.6_000.h5" 

# Select the range of bins you want to look at
start = 0
end   = 5

inspect_px5_h5(h5_file, start, end)